### 创建树
1.evt: event number

2.energy: measured total energy

3.time: measured time

4.adc[4]: ADC values from 4 channels

5.x[2]: two reconstructed x positions

6.用TVector构造的三维顶点

int 是C++的；Int_t 是ROOT的；G4doulbe 是Geant4/NPTOOL的

In [2]:
%jsroot on
TCanvas*c1=new TCanvas("c1","c1",800,600);

In [3]:
TFile*fout= new TFile("tree_basic.root","recreate");
TTree*tree=new TTree("tree","simple event tree");

Int_t evt;
Float_t energy;
Float_t t;
Int_t adc[4];
TVector3 vertex;



创建新的分支tree->Branch

读取分支是tree->SetBranchAddress

In [5]:
tree->Branch("evt",&evt,"evt/I");
tree->Branch("energy",&energy,"energy/F");
tree->Branch("t",&t,"t/F");
tree->Branch("adc",adc,"adc[4]/I");//第三个参数是数组的数据类型,数组不能加&符号！！！！！
tree->Branch("vertex",&vertex);//由于TVector3是ROOT自带的数据类型，故不用告诉ROOT数据类型

填树

In [7]:
TRandom r(0);
for(evt=0;evt<5000;evt++)
    {
        if(r.Uniform(0,1)<0.65)
            energy=r.Gaus(950,70);
        else
            energy=r.Gaus(1350,90);

        t = 20 + 0.02*energy + r.Gaus(0,4);

        adc[0] = r.Poisson(0.08*energy);
        adc[1] = r.Poisson(0.09*energy);
        adc[2] = r.Poisson(0.07*energy);
        adc[3] = r.Poisson(0.06*energy);

        double vertexX = r.Gaus(500,600);
        double vertexY = r.Gaus(300,450);
        double vertexZ = r.Gaus(300,500);

        vertex.SetXYZ(vertexX,vertexY,vertexZ);

        tree->Fill(); //在事件循环中填入
    }

In [8]:
tree->Write();
fout->Close();

ROOT文件已经创建好了，现在开始读取

In [10]:
TFile*fin=new TFile("tree_basic.root");
fin->ls();

TFile**		tree_basic.root	
 TFile*		tree_basic.root	
  KEY: TTree	tree;1	simple event tree


In [11]:
TTree*tree=(TTree*)fin->Get("tree");
tree->Print();

******************************************************************************
*Tree    :tree      : simple event tree                                      *
*Entries :     5000 : Total =          363939 bytes  File  Size =     203066 *
*        :          : Tree compression factor =   1.79                       *
******************************************************************************
*Br    0 :evt       : evt/I                                                  *
*Entries :     5000 : Total  Size=      20547 bytes  File Size  =       7080 *
*Baskets :        1 : Basket Size=      32000 bytes  Compression=   2.83     *
*............................................................................*
*Br    1 :energy    : energy/F                                               *
*Entries :     5000 : Total  Size=      20562 bytes  File Size  =      17143 *
*Baskets :        1 : Basket Size=      32000 bytes  Compression=   1.17     *
*...................................................

In [12]:
tree->Show(10);//显示第10个事件

======> EVENT:10
 evt             = 10
 energy          = 872.35
 t               = 40.7366
 adc             = 70, 
                  76, 50, 44
 vertex          = (TVector3*)0x600002b59cb0


In [13]:
tree->Scan("evt:energy:t:adc[0]:adc[1]:vertex.X():vertex.Y()","","",10,1);

************************************************************************************************
*    Row   *       evt *    energy *         t *    adc[0] *    adc[1] * vertex.X( * vertex.Y( *
************************************************************************************************
*        1 *         1 * 1002.1604 * 42.672332 *        89 *        90 * 482.52167 * 248.05309 *
*        2 *         2 * 1291.3651 * 40.172206 *        97 *       133 * 535.16477 * 129.43638 *
*        3 *         3 * 1270.9574 * 48.770248 *       105 *       129 * -319.8702 * -288.0712 *
*        4 *         4 * 809.26318 * 33.338092 *        56 *        65 * 1262.8966 * -210.0516 *
*        5 *         5 * 1022.3822 * 33.843071 *        81 *        87 *  1358.334 * -837.2961 *
*        6 *         6 * 953.10791 * 39.562107 *        74 *        79 * 245.08869 * 135.85093 *
*        7 *         7 * 791.70819 * 35.356742 *        75 *        62 * 851.02853 * 94.582342 *
*        8 *         8 * 953.3

#### 要对能谱进行拟合必须先保存为histogram对象 TH1F*h1=(TH1F*)gDirectory->Get("h1");

In [15]:
tree->Draw("energy>>h1(100,0,2000)");//要对能谱进行拟合必须先保存为histogram对象
TH1F*h1=(TH1F*)gDirectory->Get("h1");
h1->SetTitle("Energy;Energy;Counts");
c1->Draw();

常见 Fit options
R:限制拟合范围
Q:不打印过程
L:likelihood 拟合
+:不清除已有函数

In [17]:
TF1*fgauss1 = new TF1("fgauss1","[0]*exp(-0.5*(pow((x-[1])/[2],2)))");
TF1*fgauss2 = new TF1("fgauss2","[0]*exp(-0.5*(pow((x-[1])/[2],2)))");//注意定义不同的函数，系数的编号都是从0开始的

fgauss1->SetParameter(0,450);
fgauss1->SetParameter(1,950);
fgauss1->SetParameter(2,70); //必须要给定拟合初值才可以
fgauss1->SetLineColor(kRed);

fgauss2->SetParameter(0,200);
fgauss2->SetParameter(1,1350);
fgauss2->SetParameter(2,90);
fgauss2->SetLineColor(kBlue);

h1->Fit(fgauss1);
c1->Draw("same");
h1->Fit(fgauss2,"+");
c1->Draw("same");

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1762.46
NDf                       =           45
Edm                       =  4.63943e-08
NCalls                    =           72
p0                        =      367.821   +/-   8.28602     
p1                        =      953.754   +/-   1.2533      
p2                        =      70.2394   +/-   0.990754    
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      3218.24
NDf                       =           45
Edm                       =  4.77244e-09
NCalls                    =           82
p0                        =       145.07   +/-   4.65329     
p1                        =      1340.32   +/-   2.59325     
p2                        =        98.05   +/-   2.12876     


In [18]:
TF1*fgauss1 = new TF1("fgauss1","[0]*exp(-0.5*(pow((x-[1])/[2],2)))");
TF1*fgauss2 = new TF1("fgauss2","[0]*exp(-0.5*(pow((x-[1])/[2],2)))");//注意定义不同的函数，系数的编号都是从0开始的

TF1*fgauss = new TF1("fgauss","[0]*exp(-0.5*(pow((x-[1])/[2],2)))+[3]*exp(-0.5*(pow((x-[4])/[5],2)))");//好像不能分两行写

fgauss1->SetParameter(0,450);
fgauss1->SetParameter(1,950);
fgauss1->SetParameter(2,70); //必须要给定拟合初值才可以
fgauss1->SetLineColor(kRed);

fgauss2->SetParameter(0,200);
fgauss2->SetParameter(1,1350);
fgauss2->SetParameter(2,90);
fgauss2->SetLineColor(kBlue);

fgauss->SetParameters(450,950,70,200,1350,90);//注意是SetParameters后面有个s
fgauss->SetLineColor(kGreen);

h1->Fit(fgauss1);
c1->Draw("same");
h1->Fit(fgauss2,"+");
c1->Draw("same");

h1->Fit(fgauss,"+");
c1->Draw("same");

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1762.46
NDf                       =           45
Edm                       =  4.63943e-08
NCalls                    =           72
p0                        =      367.821   +/-   8.28602     
p1                        =      953.754   +/-   1.2533      
p2                        =      70.2394   +/-   0.990754    
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      3218.24
NDf                       =           45
Edm                       =  4.77244e-09
NCalls                    =           82
p0                        =       145.07   +/-   4.65329     
p1                        =      1340.32   +/-   2.59325     
p2                        =        98.05   +/-   2.12876     
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      44.6116
NDf                       =           42
Edm  

#### 前面的数组是固定值，但实际上每个事件中的对象数量往往是变化的，例如探测器中的击中（hits）数目 

#### ROOT 支持变长数组，用一个整数分支存储该事件中对象的个数，用另一个数组分支存储数据，而该数组的有效长度由这个整数决定

In [20]:
TFile*fhit = new TFile("tree_hits.root","recreate");
TTree*tree = new TTree("tree","event tree with variable-length hit arrays");

Int_t evt;
Int_t nhit;
Float_t xpos[10];
Float_t ypos[10];
Float_t e[10];

tree->Branch("evt",  &evt,  "evt/I");
tree->Branch("nhit", &nhit, "nhit/I");
tree->Branch("xpos", xpos,  "xpos[nhit]/F");
tree->Branch("ypos", ypos,  "ypos[nhit]/F");  //注意数组不能加&，&是指向地址的，数组已经有地址了。常数和向量要加&
tree->Branch("e", e,  "e[nhit]/F");

TRandom3 r1(0);

//外层是事件循环，内层是击中数
//对于一个事件可能有多次击中，如，第一个事件产生了三次击中，就会有三个坐标
for(evt=0;evt<5000;evt++)
    {
        nhit=r1.Integer(5); //击中的数目为0，1，2，3，4

        for(int i=0;i<nhit;i++)
            {
                xpos[i]=r1.Gaus(0,10);
                ypos[i] = r1.Gaus(0,8);
                e[i] = r1.Exp(2.0);
            }

        tree->Fill();//在事件循环中填入
    }

tree->Write();
fhit->Close();

#### 读取树的操作

In [3]:
//使用 tree->SetBranchAddress
//用于读取事件的个数 tree->GetEntriesFast();
TFile *fin = new TFile("tree_basic.root");

Int_t evt_r;
Float_t energy_r, t_r;
Int_t adc_r[4];

tree->SetBranchAddress("evt",    &evt_r);
tree->SetBranchAddress("energy", &energy_r);
tree->SetBranchAddress("t",      &t_r);
tree->SetBranchAddress("adc",    adc_r);//同样，由于adc是数组，所以不用加&

TH1F *hsel = new TH1F("hsel","Selected energy;Energy;Counts",100,700,1600);

Long64_t nentries = tree->GetEntriesFast();//用于读取事件的个数

for(Long64_t i=0; i<nentries; i++) 
{
    tree->GetEntry(i);

    if(t_r > 45 && adc_r[0] > 70) 
    {
        hsel->Fill(energy_r);
    }
}

hsel->Draw();
c1->Draw();

### 用Macro读取TVector3 对应生成test_tvector3.root的宏文件是make_tvector3.C

In [5]:
TFile *f = new TFile("test_tvector3.root", "READ");
TTree *t = (TTree*)f->Get("tree");

TVector3 *pos = nullptr;  //这就是指针，要和make_tvector3.C对应上
TVector3 *vertex = nullptr;

t->SetBranchAddress("pos", &pos);
t->SetBranchAddress("vertex",&vertex);

TH1F *hTheta = new TH1F("hTheta", "Theta of TVector3;#theta [rad];Counts", 100, 0, 3.2);
TH2F* hvertex =new TH2F("hvertex","Vertx distribution;#x;#y",120,-30,30,120,-30,30);

Long64_t nentries = t->GetEntriesFast();

for (Long64_t i = 0; i < nentries; i++)
{
    t->GetEntry(i);

    double thetaValue = pos->Theta();
    hTheta->Fill(thetaValue);

    double vertexX = vertex->X();
    double vertexY = vertex->Y();
    hvertex->Fill(vertexX,vertexY);
}

c1->Divide(2,1);
c1->cd(1);

hTheta->Draw();
c1->cd(2);
hvertex->Draw();
c1->Draw();


### 使用TTreeReader 

In [20]:
TFile *f = new TFile("test_tvector3.root", "READ");

TTreeReader reader("tree", f);

TTreeReaderValue<TVector3> pos(reader, "pos");
TTreeReaderValue<TVector3> vertex(reader, "vertex");

TH1F *hTheta = new TH1F("hTheta","Theta of TVector3;#theta [rad];Counts",100, 0, 3.2);
TH2F *hvertex = new TH2F("hvertex","Vertex distribution;x;y",120, -30, 30,120, -30, 30);

while (reader.Next()) 
{
    double thetaValue = pos->Theta();
    hTheta->Fill(thetaValue);

    double vertexX = vertex->X();
    double vertexY = vertex->Y();
    hvertex->Fill(vertexX, vertexY);
}

TCanvas *c3 = new TCanvas("c1", "c1", 1000,800);

c3->Divide(2, 1);

c3->cd(1);
hTheta->Draw();

c3->cd(2);
hvertex->Draw("colz");

c3->Draw();


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


### RDataFrame 有点复杂！

In [29]:
ROOT::RDataFrame df("tree", "test_tvector3.root");

auto df2 = df
    .Define("theta",  [](const TVector3& p) { return p.Theta(); }, {"pos"})
    .Define("vertexX", [](const TVector3& v) { return v.X(); }, {"vertex"})
    .Define("vertexY", [](const TVector3& v) { return v.Y(); }, {"vertex"});

auto hTheta = df2.Histo1D(
    {"hTheta", "Theta of TVector3;#theta [rad];Counts", 100, 0, 3.2},
    "theta"
);

auto hvertex = df2.Histo2D(
    {"hvertex", "Vertex distribution;x;y", 120, -30, 30, 120, -30, 30},
    "vertexX",
    "vertexY"
);

TCanvas *c4 = new TCanvas("c1", "c1", 1000,800);

c4->Divide(2, 1);

c4->cd(1);
hTheta->Draw();

c4->cd(2);
hvertex->Draw("colz");

c4->Draw();

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
